# فصل ۱۱: تعبیه متن (Text Embeddings)
## بخش ۱: پیاده‌سازی عملی مدل‌های تعبیه متن و سنجش شباهت در LangChain

### مفاهیم کلیدی:
1. **بردار تعبیه (Embedding Vector):** تبدیل متن به یک بردار عددی چندبعدی (مثلاً ۷۶۸ یا ۱۵۳۶ بعد) که معنا و مفهوم معنایی (Semantic Meaning) متن را نمایش می‌دهد.
2. **شباهت کسینوسی (Cosine Similarity):** معیاری بین `-1` تا `1` (معمولاً در تعبیه‌ها بین `0` تا `1`) که زاویه بین دو بردار را می‌سنجد. هر چه به `1` نزدیک‌تر باشد، دو متن هم‌معنی‌ترند و شباهت یک متن با خودش دقیقاً `1.0` است.
3. **دو متد اصلی در LangChain:**
   - `embed_query(text)`: بردارسازی یک رشته متنی منفرد (مناسب برای پرسش و کوئری جستجو).
   - `embed_documents([doc1, doc2, ...])`: بردارسازی دسته‌ای لیستی از اسناد و متون.

In [7]:
pip install langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import urllib.request
from dotenv import load_dotenv
import numpy as np

# ۱. ابتدا بارگذاری متغیرهای محیطی از فایل .env
load_dotenv()

# ۲. حذف پروکسی‌های قدیمی و غیرفعال محیطی برای جلوگیری از خطای WinError 10061
for var in [
    "http_proxy",
    "https_proxy",
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "all_proxy",
    "ALL_PROXY",
]:
  os.environ.pop(var, None)

# غیرفعال کردن کامل پروکسی برای اطمینان از عدم خوانش تنظیمات مخفی
os.environ["NO_PROXY"] = "*"
os.environ["no_proxy"] = "*"
urllib.request.getproxies = lambda: {}

# ۳. تابع سنجش شباهت کسینوسی (سازگار با sklearn و دارای جایگزین بومی numpy)
try:
  from sklearn.metrics.pairwise import cosine_similarity
except ImportError:

  def cosine_similarity(a, b):
    """محاسبه کسینوس شباهت با numpy بدون نیاز به کتابخانه sklearn"""
    a = np.asarray(a)
    b = np.asarray(b)
    dot_val = np.dot(a, b.T)
    norm_a = np.linalg.norm(a, axis=-1, keepdims=True)
    norm_b = np.linalg.norm(b, axis=-1, keepdims=True)
    return dot_val / (norm_a * norm_b.T)


print("✅ تنظیمات اولیه و تابع cosine_similarity بارگذاری شد!")

✅ تنظیمات اولیه و تابع cosine_similarity بارگذاری شد!


### ۱. مدل‌های تجاری و ابری (Cloud / API Models)

در پروژه‌ها می‌توانید بر حسب نیاز از مدل‌های گوگل جمینای یا مدل‌های OpenAI/تپ‌سیج استفاده کنید.
در اینجا مدل **`GoogleGenerativeAIEmbeddings`** (مدل `text-embedding-004`) با کلید `.env` فعلی شما آماده کار است، و کد معادل OpenAI نیز قرار داده شده است:

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

# دانلود و بارگذاری مدل چندزبانه (پشتیبانی عالی از فارسی و بدون نیاز به فیلترشکن یا API Key)
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ مدل تعبیه متن HuggingFace با موفقیت بارگذاری شد!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1038.28it/s]


✅ مدل تعبیه متن HuggingFace با موفقیت بارگذاری شد!


### ۲. استخراج بردار تعبیه با متد `embed_query`

سه جمله زیر را بررسی می‌کنیم:
- `text1`: «فوتبال بهترین ورزش است.»
- `text2`: «برترین ورزش فوتبال است.» (کاملاً هم‌معنی با جمله اول اما با کلمات متفاوت)
- `text3`: «هوا بسیار سرد شده است.» (موضوعی کاملاً بی‌ربط)

In [10]:
text1 = "فوتبال بهترین ورزش است."
text2 = "برترین ورزش فوتبال است."
text3 = "هوا بسیار سرد شده است."

# دریافت بردارهای تعبیه به صورت تک‌متن
embed1 = embeddings_model.embed_query(text1)
embed2 = embeddings_model.embed_query(text2)
embed3 = embeddings_model.embed_query(text3)

print(f"تعداد ابعاد بردار تعبیه: {len(embed1)}")
print(f"نمونه ۱۰ عدد اول بردار: {embed1[:10]}")

تعداد ابعاد بردار تعبیه: 384
نمونه ۱۰ عدد اول بردار: [0.005599692929536104, -0.09677659720182419, -0.200890451669693, -0.27681371569633484, 0.3567775785923004, 0.3526356816291809, 0.26425573229789734, 0.2712356150150299, 0.2595597803592682, 0.24477966129779816]


### ۳. محاسبه و سنجش شباهت کسینوسی (Cosine Similarity)

با تبدیل لیست‌ها به آرایه‌های دو بعدی (`reshape(1, -1)`)، شباهت معنایی جملات را می‌سنجیم:

In [11]:
embed1_array = np.array(embed1).reshape(1, -1)
embed2_array = np.array(embed2).reshape(1, -1)
embed3_array = np.array(embed3).reshape(1, -1)

similarity12 = cosine_similarity(embed1_array, embed2_array)[0][0]
similarity13 = cosine_similarity(embed1_array, embed3_array)[0][0]
similarity23 = cosine_similarity(embed2_array, embed3_array)[0][0]
similarity11 = cosine_similarity(embed1_array, embed1_array)[0][0]

print(f"Cosine Similarity 1 & 2 (فوتبال و فوتبال): {similarity12:.4f}")
print(f"Cosine Similarity 1 & 3 (فوتبال و هوا):     {similarity13:.4f}")
print(f"Cosine Similarity 2 & 3 (ورزش و هوا):       {similarity23:.4f}")
print(f"Cosine Similarity 1 & 1 (شباهت با خود):    {similarity11:.4f}")

Cosine Similarity 1 & 2 (فوتبال و فوتبال): 0.8296
Cosine Similarity 1 & 3 (فوتبال و هوا):     -0.0539
Cosine Similarity 2 & 3 (ورزش و هوا):       -0.0367
Cosine Similarity 1 & 1 (شباهت با خود):    1.0000


### ۴. بردارسازی دسته‌ای با `embed_documents`

زمانی که تعداد زیادی متن یا سند داریم، به جای حلقه و فراخوانی مکرر `embed_query`، از متد **`embed_documents`** استفاده می‌کنیم تا در یک درخواست دسته‌ای (Batch) با سرعت بالا همه را بردارسازی کند:

In [12]:
texts = [text1, text2, text3]
embeddings = embeddings_model.embed_documents(texts)

print(f"تعداد متون پردازش‌شده: {len(embeddings)}")
print(f"طول بعد هر بردار: {len(embeddings[0])}")
print(f"نمونه اعداد بردار اول: {embeddings[0][:5]}")

تعداد متون پردازش‌شده: 3
طول بعد هر بردار: 384
نمونه اعداد بردار اول: [0.0055997492745518684, -0.09677649289369583, -0.20089051127433777, -0.2768135964870453, 0.35677751898765564]


### ۵. مدل‌های متن‌باز و لوکال با Ollama (`bge-m3`)

اگر بخواهید بدون وابستگی به اینترنت یا هزینه API، تعبیه‌ها را روی سیستم خود بسازید، مدل چندزبانه **`bge-m3`** از طریق ابزار **Ollama** یکی از بهترین گزینه‌ها به خصوص برای زبان فارسی است.

#### مراحل استقرار مدل در ترمینال:
```bash
# دانلود مدل تعبیه متن bge-m3 در Ollama
ollama pull bge-m3
```

#### استفاده در پایتون با `langchain_ollama`:
```python
from langchain_ollama import OllamaEmbeddings

local_embeddings = OllamaEmbeddings(model="bge-m3")
text = "من کدنویسی را دوست دارم."
embed = local_embeddings.embed_query(text)
print(f"تعداد ابعاد بردار bge-m3: {len(embed)}")  # 1024 بعد
```

--- 

### جدول پارامترهای تنظیمی کلاس `OllamaEmbeddings`:

| نام پارامتر | نوع داده / مقدار پیش‌فرض | توضیحات و کاربرد |
| :--- | :--- | :--- |
| `model` | `str` (الزامی) | نام مدل دانلود شده در Ollama (مثلاً `"bge-m3"`). |
| `base_url` | `str` / `"http://localhost:11434"` | آدرس سرور Ollama. |
| `num_ctx` | `int` / `2048` | طول پنجره بافتار. مدل `bge-m3` تا `8192` توکن را پشتیبانی می‌کند. اگر متون شما طولانی است، این مقدار را مثلاً `8192` قرار دهید. |
| `num_gpu` | `int` / `-1` | تعداد لایه‌هایی که روی GPU لود می‌شوند (`-1` بیشترین مقدار ممکن را به صورت خودکار انتخاب می‌کند). |
| `num_thread` | `int` / خودکار | تعداد هسته‌های CPU که در صورت عدم استفاده از کارت گرافیک فعال می‌شوند. |
| `keep_alive` | `str` / `"5m"` | مدت زمانی که مدل پس از آخرین درخواست در RAM/VRAM باقی می‌ماند (مثلاً `"1h"` مانع لود و آنلود مکرر می‌شود). |
| `show_progress`| `bool` / `False` | نمایش نوار پیشرفت هنگام پردازش‌های سنگین. |